# MINI Cells — Experiment 001: Echo
Thin orchestration notebook. All model and training logic is imported from `minicells`.

In [ ]:
import platform, sys, torch
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'research').exists(): ROOT = Path('/kaggle/working/mini-cells')
sys.path.insert(0, str(ROOT / 'research'))
print(platform.python_version(), torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from minicells.config import load_config
from minicells.data import CopyDataGenerator
from minicells.model import EchoModel
from minicells.ops import architecture_stats
from minicells.train import train
from minicells.vocab import CharVocab
config = load_config(ROOT / 'configs/echo-v0.yaml')
vocab = CharVocab()
generator = CopyDataGenerator(vocab, seed=config['train']['seed'], min_length=1, max_length=32, num_cells=64)
print(*generator.batch(10).texts, sep='\n')
model = EchoModel(vocab_size=len(vocab), **{k:v for k,v in config['model'].items() if k != 'vocab_size'})
print(model)
print(architecture_stats(model))

In [ ]:
report = train(config)
report

## Three-seed validation and decision
Run the next cell to produce independent seed bundles. The experiment remains INCOMPLETE until the aggregate gate is evaluated from all three real runs.

In [ ]:
from copy import deepcopy
from minicells.aggregate import aggregate_seed_runs
reports = []
for seed in (1, 2, 3):
    run = deepcopy(config)
    run['train']['seed'] = seed
    run['output']['root'] = f'results/echo-v0/seed-{seed}'
    reports.append(train(run))
decision = aggregate_seed_runs('results/echo-v0')
decision